## Example 1: tinycss2 functionality test 1

In [ ]:
import tinycss2

# Example 1: Basic CSS parsing and tokenization
print("Basic CSS parsing with tinycss2:")

# Simple CSS stylesheet
css_code = """
.container {
    color: blue;
    margin: 10px;
    background-color: #ffffff;
}

#header {
    font-size: 18px;
    padding: 5px 10px;
}
"""

# Parse the stylesheet
tokens = tinycss2.parse_stylesheet(css_code)
print(f"Parsed {len(tokens)} top-level tokens")

# Examine the tokens
for i, token in enumerate(tokens):
    if hasattr(token, 'type'):
        print(f"Token {i}: {token.type}")
    else:
        print(f"Token {i}: {type(token).__name__}")

# Parse a simple declaration list
declarations = "color: red; font-size: 16px; margin: 0"
decl_tokens = tinycss2.parse_declaration_list(declarations)
print(f"\nDeclaration parsing: {len(decl_tokens)} declarations")

for decl in decl_tokens:
    if hasattr(decl, 'name'):
        print(f"  {decl.name}: {tinycss2.serialize(decl.value)}")

print(f"TinyCSS2 version: {getattr(tinycss2, 'VERSION', 'Unknown')}")

## Example 2: tinycss2 functionality test 2

In [ ]:
import tinycss2

# Example 2: Advanced CSS parsing with error handling
print("Advanced CSS parsing with error handling:")

# CSS with various syntax elements
complex_css = """
@media (max-width: 768px) {
    .responsive { display: none; }
}

.animation {
    animation: slideIn 0.5s ease-in-out;
    transform: translate3d(0, 0, 0);
}

/* Invalid CSS for error testing */
.invalid { color: ; margin 10px }
"""

# Parse and analyze the complex CSS
tokens = tinycss2.parse_stylesheet(complex_css)
print(f"Complex CSS tokens: {len(tokens)}")

for i, token in enumerate(tokens):
    token_type = type(token).__name__
    print(f"Token {i}: {token_type}")
    
    # Check for at-rules
    if hasattr(token, 'at_keyword'):
        print(f"  At-rule: {token.at_keyword}")
    
    # Check for qualified rules (selectors + declarations)
    if hasattr(token, 'prelude') and hasattr(token, 'content'):
        selector = tinycss2.serialize(token.prelude).strip()
        print(f"  Selector: {selector[:30]}...")

# Parse individual components
single_value = "rgba(255, 0, 0, 0.5)"
component = tinycss2.parse_one_component_value(single_value)
print(f"\nSingle component parsing:")
print(f"Input: {single_value}")
print(f"Parsed: {type(component).__name__}")
print(f"Serialized: {tinycss2.serialize([component])}")

# Test error handling
invalid_css = "{ color: ; }"
try:
    error_tokens = tinycss2.parse_stylesheet(invalid_css)
    print(f"\nError handling: {len(error_tokens)} tokens (parsing continues despite errors)")
except Exception as e:
    print(f"Error handling failed: {e}")
    raise

## Example 3: tinycss2 functionality test 3

In [ ]:
import tinycss2

# Example 3: CSS serialization and round-trip processing
print("CSS serialization and round-trip:")

# Original CSS
original_css = """
.button {
    background: linear-gradient(45deg, #ff6b6b, #4ecdc4);
    border-radius: 5px;
    padding: 10px 20px;
    transition: all 0.3s ease;
}
"""

# Parse and serialize back
tokens = tinycss2.parse_stylesheet(original_css)
serialized = tinycss2.serialize(tokens)

print(f"Original CSS length: {len(original_css)} chars")
print(f"Serialized CSS length: {len(serialized)} chars")
print(f"Round-trip preservation: {original_css.strip() == serialized.strip()}")
assert serialized == original_css, "parse_stylesheet + serialize did not round-trip the stylesheet"

print("\nOriginal:")
print(original_css.strip())
print("\nSerialized:")
print(serialized.strip())

# Parse and serialize individual declarations
declarations = [
    "color: #333333",
    "font-family: 'Arial', sans-serif",
    "box-shadow: 0 2px 4px rgba(0,0,0,0.1)"
]

print(f"\nDeclaration processing:")
for decl in declarations:
    parsed = tinycss2.parse_one_declaration(decl)
    # a parse failure is returned (not raised) as a ParseError node, which is truthy — check its type
    assert parsed.type == "declaration", f"failed to parse {decl!r}: {getattr(parsed, 'message', parsed)}"
    reserialized = f"{parsed.name}: {tinycss2.serialize(parsed.value)}"
    print(f"'{decl}' → '{reserialized}'")

# Test identifier serialization
identifiers = ["button", "nav-menu", "header_title", "123invalid"]
print(f"\nIdentifier serialization:")
for ident in identifiers:
    try:
        serialized_ident = tinycss2.serialize_identifier(ident)
        print(f"'{ident}' → '{serialized_ident}'")
    except Exception as e:
        print(f"'{ident}' → Error: {e}")
        raise

## Example 4: tinycss2 functionality test 4

In [ ]:
import tinycss2

# Example 4: CSS parsing utilities and component extraction
print("CSS parsing utilities and component extraction:")

# CSS with various components
mixed_css = """
/* Variables and custom properties */
:root {
    --primary-color: #007bff;
    --font-size: 16px;
}

/* Keyframes animation */
@keyframes fadeIn {
    from { opacity: 0; }
    to { opacity: 1; }
}

/* Complex selectors and properties */
.card:hover > .title::before {
    content: "→ ";
    color: var(--primary-color);
    font-size: calc(var(--font-size) + 2px);
}
"""

# Parse the mixed CSS
tokens = tinycss2.parse_stylesheet(mixed_css)
print(f"Mixed CSS analysis:")

for i, token in enumerate(tokens):
    if hasattr(token, 'type'):
        print(f"Token {i}: {token.type}")
    elif hasattr(token, 'at_keyword'):
        print(f"Token {i}: At-rule '{token.at_keyword}'")
        if token.at_keyword == 'keyframes':
            print(f"  Animation name: {tinycss2.serialize(token.prelude).strip()}")
    elif hasattr(token, 'prelude'):
        selector = tinycss2.serialize(token.prelude).strip()
        print(f"Token {i}: Rule with selector '{selector[:50]}...'")

# Parse component values list
values_list = "10px 20px 30px, rgba(255, 0, 0, 0.5), 'Arial', sans-serif"
components = tinycss2.parse_component_value_list(values_list)
print(f"\nComponent value list parsing:")
print(f"Input: {values_list}")
print(f"Components found: {len(components)}")

for i, comp in enumerate(components):
    comp_type = type(comp).__name__
    serialized = tinycss2.serialize([comp])
    print(f"  Component {i}: {comp_type} → '{serialized.strip()}'")

# Parse rule list
rules_text = """
.header { background: blue; }
.footer { background: gray; }
"""
rules = tinycss2.parse_rule_list(rules_text)
print(f"\nRule list parsing: {len(rules)} rules found")

for rule in rules:
    if hasattr(rule, 'prelude') and hasattr(rule, 'content'):
        selector = tinycss2.serialize(rule.prelude).strip()
        print(f"  Rule: {selector}")

## Example 5: tinycss2 functionality test 5

In [ ]:
import tinycss2

# Example 5: CSS color extraction and minification
print("CSS color extraction and minification:")

# Sample CSS with colors
css_with_colors = """
.header {
    background-color: #3498db;
    color: white;
    border: 1px solid #2980b9;
}

.button {
    background: rgba(255, 0, 0, 0.5);
    color: hsl(120, 100%, 50%);
}
"""

# Extract colors from CSS
print("Color extraction:")
tokens = tinycss2.parse_stylesheet(css_with_colors)

for token in tokens:
    if hasattr(token, 'content'):
        declarations = tinycss2.parse_declaration_list(tinycss2.serialize(token.content))
        for decl in declarations:
            if hasattr(decl, 'name') and hasattr(decl, 'value'):
                value_str = tinycss2.serialize(decl.value)
                if any(c in value_str for c in ['#', 'rgb', 'hsl']):
                    print(f"  {decl.name}: {value_str.strip()}")

# Minify CSS
print("\nCSS minification:")
original = tinycss2.serialize(tokens)
print(f"Original size: {len(css_with_colors)} chars")
print(f"Parsed and serialized: {len(original)} chars")

print("\nExample 5 completed!")